# Домашнє завдання 7. RAG-система для аналізу фінансових звітів

Система відповідає на питання про фінансові показники компаній, використовуючи векторне сховище Chroma та LLM через OpenRouter.

### Встановлення залежностей

In [ ]:
!pip install --no-warn-conflicts "langchain>=0.3.30,<0.4.0" "langchain-core>=0.3.85,<0.4.0" "opentelemetry-api==1.42.1" "opentelemetry-sdk==1.42.1"
!pip install --no-warn-conflicts langchain-community langchain-text-splitters langchain-openai chromadb tiktoken
import warnings
warnings.filterwarnings("ignore")

  Using cached langchain_core-0.3.86-py3-none-any.whl.metadata (3.2 kB)
  Using cached langchain_text_splitters-0.3.11-py3-none-any.whl.metadata (1.8 kB)
Using cached langchain_core-0.3.86-py3-none-any.whl (461 kB)
Using cached langchain_text_splitters-0.3.11-py3-none-any.whl (33 kB)
  Attempting uninstall: opentelemetry-api
    Found existing installation: opentelemetry-api 1.45.2
    Uninstalling opentelemetry-api-1.45.2:
      Successfully uninstalled opentelemetry-api-1.45.2
  Attempting uninstall: opentelemetry-semantic-conventions
    Found existing installation: opentelemetry-semantic-conventions 0.66b2
    Uninstalling opentelemetry-semantic-conventions-0.66b2:
      Successfully uninstalled opentelemetry-semantic-conventions-0.66b2
  Attempting uninstall: opentelemetry-sdk
    Found existing installation: opentelemetry-sdk 1.45.2
    Uninstalling opentelemetry-sdk-1.45.2:
      Successfully uninstalled opentelemetry-sdk-1.45.2
  Attempting uninstall: langchain-core
    Found e

### Налаштування змінних середовища

In [ ]:
import os
from getpass import getpass

def get_secret(name: str) -> str:
    # 1) змінна середовища, 2) Colab Secrets, 3) ручне введення
    if os.environ.get(name):
        return os.environ[name]
    try:
        from google.colab import userdata
        value = userdata.get(name)
        if value:
            return value
    except Exception:
        pass
    return getpass(f"Введіть {name}: ")

# Ключ OpenRouter (у ноутбуці не зберігається)
os.environ["OPENROUTER_API_KEY"] = get_secret("OPENROUTER_API_KEY")

BASE_URL = "https://openrouter.ai/api/v1"
MODEL = "openai/gpt-4o-mini"
EMBEDDING_MODEL = "openai/text-embedding-3-small"

Введіть OPENROUTER_API_KEY: ··········


### Вхідні дані

In [ ]:
financial_data = """ТехНова Інк за 2024 рік: виручка 450 млн дол (+18 відсотків), чистий прибуток 67 млн дол, маржа EBITDA 22 відсотки.
Працівників 2300 осіб. ROE становить 15.2 відсотка. Співвідношення борг до EBITDA дорівнює 1.8. Штаб-квартира у Львові.

ГрінЕнерджі Солюшнс за 2024 рік: виручка 890 млн дол (+35 відсотків), чистий прибуток 156 млн дол, маржа EBITDA 28 відсотків.
Працівників 5100 осіб. ROE становить 19.7 відсотка. Співвідношення борг до EBITDA дорівнює 0.9. Лідер у секторі відновлюваної енергії.

РітейлМакс Груп за 2024 рік: виручка 1200 млн дол (+8 відсотків), чистий прибуток 48 млн дол, маржа EBITDA 12 відсотків.
Працівників 8500 осіб. ROE становить 9.3 відсотка. Співвідношення борг до EBITDA дорівнює 3.2. Мережа з 340 магазинів в Україні.

ФінТех Дайнамікс за 2024 рік: виручка 230 млн дол (+52 відсотки), чистий збиток мінус 15 млн дол, маржа EBITDA мінус 8 відсотків.
Працівників 890 осіб. Від'ємний показник ROE. Стартап який активно інвестує у масштабування бізнесу."""

### Розбиття тексту на чанки

Компанії у тексті розділені порожнім рядком, тому розбиваю по `\n\n`. Опис однієї компанії займає приблизно 250 символів, тому `chunk_size=300` — так один опис вміщується повністю, а два сусідні описи не зливаються в один чанк.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n"],
    chunk_size=300,
    chunk_overlap=0,
)

# Кожна компанія стає окремим документом
splits = text_splitter.create_documents([financial_data])

print(f"Кількість чанків: {len(splits)}\n")
for i, doc in enumerate(splits, start=1):
    print(f"--- Чанк {i} ({len(doc.page_content)} символів) ---")
    print(doc.page_content)
    print()

Кількість чанків: 4

--- Чанк 1 (235 символів) ---
ТехНова Інк за 2024 рік: виручка 450 млн дол (+18 відсотків), чистий прибуток 67 млн дол, маржа EBITDA 22 відсотки.
Працівників 2300 осіб. ROE становить 15.2 відсотка. Співвідношення борг до EBITDA дорівнює 1.8. Штаб-квартира у Львові.

--- Чанк 2 (260 символів) ---
ГрінЕнерджі Солюшнс за 2024 рік: виручка 890 млн дол (+35 відсотків), чистий прибуток 156 млн дол, маржа EBITDA 28 відсотків.
Працівників 5100 осіб. ROE становить 19.7 відсотка. Співвідношення борг до EBITDA дорівнює 0.9. Лідер у секторі відновлюваної енергії.

--- Чанк 3 (249 символів) ---
РітейлМакс Груп за 2024 рік: виручка 1200 млн дол (+8 відсотків), чистий прибуток 48 млн дол, маржа EBITDA 12 відсотків.
Працівників 8500 осіб. ROE становить 9.3 відсотка. Співвідношення борг до EBITDA дорівнює 3.2. Мережа з 340 магазинів в Україні.

--- Чанк 4 (230 символів) ---
ФінТех Дайнамікс за 2024 рік: виручка 230 млн дол (+52 відсотки), чистий збиток мінус 15 млн дол, маржа EBITD

### Векторне сховище

In [ ]:
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(
    model=EMBEDDING_MODEL,
    openai_api_base=BASE_URL,
    openai_api_key=os.environ.get("OPENROUTER_API_KEY"),
)

# Ембедінг чанків та збереження у колекції Chroma
# ids потрібні, щоб при повторному запуску клітинки документи не дублювалися
vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embeddings,
    collection_name="finansovi_zvity_kompaniy",
    ids=[f"company_{i}" for i in range(1, len(splits) + 1)],
)

print(f"Документів у колекції: {vectorstore._collection.count()}")

Документів у колекції: 4


### Налаштування ретрівера

In [ ]:
# Ретрівер повертає 2 найбільш релевантні чанки
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

# Перевірка пошуку
found_docs = retriever.invoke("Яка виручка у компанії ТехНова Інк за 2024 рік?")
print(f"Знайдено документів: {len(found_docs)}\n")
for doc in found_docs:
    print(doc.page_content)
    print()

Знайдено документів: 2

ТехНова Інк за 2024 рік: виручка 450 млн дол (+18 відсотків), чистий прибуток 67 млн дол, маржа EBITDA 22 відсотки.
Працівників 2300 осіб. ROE становить 15.2 відсотка. Співвідношення борг до EBITDA дорівнює 1.8. Штаб-квартира у Львові.

ФінТех Дайнамікс за 2024 рік: виручка 230 млн дол (+52 відсотки), чистий збиток мінус 15 млн дол, маржа EBITDA мінус 8 відсотків.
Працівників 890 осіб. Від'ємний показник ROE. Стартап який активно інвестує у масштабування бізнесу.



### Промпт та модель

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

template = """Ти асистент, який відповідає на питання про фінансові показники компаній.
Використовуй лише наведений нижче контекст. Відповідай українською мовою, коротко, з конкретними цифрами з контексту.
Якщо в контексті немає інформації про компанію з питання, так і скажи, що інформація відсутня. Не використовуй дані інших компаній і нічого не вигадуй.

Контекст:
{context}

Питання: {question}

Відповідь:"""

prompt = ChatPromptTemplate.from_template(template)

llm = ChatOpenAI(
    api_key=os.environ.get("OPENROUTER_API_KEY"),
    base_url=BASE_URL,
    model=MODEL,
    temperature=0,
)

### RAG ланцюг

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Об'єднання знайдених документів в один рядок контексту
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

### Тестування

In [ ]:
questions = [
    "Яка виручка у компанії ТехНова Інк за 2024 рік?",
    "Скільки працівників у ГрінЕнерджі Солюшнс?",
    "Який показник ROE у РітейлМакс Груп?",
    "Де знаходиться штаб-квартира ТехНова Інк?",
    "Яка маржа EBITDA у ФінТех Дайнамікс?",
    "Який чистий прибуток у ЕкоПакування Солюшнс?",
]

print("=" * 70)
for i, question in enumerate(questions, start=1):
    answer = rag_chain.invoke(question)
    print(f"\nПитання {i}: {question}")
    print(f"Відповідь: {answer}")
    print("-" * 70)


Питання 1: Яка виручка у компанії ТехНова Інк за 2024 рік?
Відповідь: Виручка компанії ТехНова Інк за 2024 рік становить 450 млн дол.
----------------------------------------------------------------------

Питання 2: Скільки працівників у ГрінЕнерджі Солюшнс?
Відповідь: У ГрінЕнерджі Солюшнс працює 5100 осіб.
----------------------------------------------------------------------

Питання 3: Який показник ROE у РітейлМакс Груп?
Відповідь: ROE у РітейлМакс Груп становить 9.3 відсотка.
----------------------------------------------------------------------

Питання 4: Де знаходиться штаб-квартира ТехНова Інк?
Відповідь: Штаб-квартира ТехНова Інк знаходиться у Львові.
----------------------------------------------------------------------

Питання 5: Яка маржа EBITDA у ФінТех Дайнамікс?
Відповідь: Маржа EBITDA у ФінТех Дайнамікс становить мінус 8 відсотків.
----------------------------------------------------------------------

Питання 6: Який чистий прибуток у ЕкоПакування Солюшнс?
Відпові

### Висновок

Текст розбито на 4 чанки, по одному на компанію. Ретрівер повертає 2 найближчі чанки, і модель відповідає тільки на їх основі. На питання про компанію, якої немає в даних, система повідомляє, що інформація відсутня.